# GRU — Gated Recurrent Unit (2014)_Papers · Sequence & NLP_**A simpler gated recurrent cell: two gates, one state, no separate memory cell — yet it still bridges long time gaps.**---This notebook is a practice scaffold for the **GRU — Gated Recurrent Unit (2014)** lesson. We rebuild the GRU's four equations by hand, check them against PyTorch's `nn.GRUCell`, roll the cell over a sequence, and then watch a GRU outlast a plain RNN across a widening memory gap. Run each cell, read the note above it, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorchWe rebuild the GRU one step at a time: (1) unpack PyTorch's weights and write the four GRU equations by hand, checking them against `nn.GRUCell`, (2) roll the hand-built cell over a whole sequence, and (3) recompute the lesson's scalar worked example to confirm the numbers.

### Step 1 — Unpack the weights and write the four GRU equations`nn.GRUCell` packs its weights for all three gates (reset `r`, update `z`, candidate `n`) into single tensors, in that order. We slice them apart so each gate gets its own weight and bias. Then we implement the four equations directly: the **reset** gate, the **update** gate, the **candidate** state (where the reset gate gates only the hidden contribution), and PyTorch's blend `(1-z)*n + z*h`. Matching `nn.GRUCell` to within floating-point error confirms we have the formula and the weight packing right.

In [ ]:
import torch
import torch.nn as nn

torch.manual_seed(0)

in_dim, hid = 3, 4
cell = nn.GRUCell(in_dim, hid)          # the reference we must match

# Pull PyTorch's packed weights, sliced in order [r, z, n].
Wi = cell.weight_ih          # (3*hid, in_dim)
Wh = cell.weight_hh          # (3*hid, hid)
bi = cell.bias_ih            # (3*hid,)
bh = cell.bias_hh            # (3*hid,)

Wir, Wiz, Win = Wi[:hid], Wi[hid:2*hid], Wi[2*hid:]
Whr, Whz, Whn = Wh[:hid], Wh[hid:2*hid], Wh[2*hid:]
bir, biz, bin = bi[:hid], bi[hid:2*hid], bi[2*hid:]
bhr, bhz, bhn = bh[:hid], bh[hid:2*hid], bh[2*hid:]

def my_gru(x, h):                                  # the four GRU equations, by hand
    r = torch.sigmoid(x @ Wir.T + bir + h @ Whr.T + bhr)          # reset gate
    z = torch.sigmoid(x @ Wiz.T + biz + h @ Whz.T + bhz)          # update gate
    n = torch.tanh(x @ Win.T + bin + r * (h @ Whn.T + bhn))       # candidate; r gates hidden part only
    return (1 - z) * n + z * h                                    # PyTorch blend

x = torch.randn(2, in_dim)
h0 = torch.randn(2, hid)
mine = my_gru(x, h0)
ref = cell(x, h0)
print("allclose:", torch.allclose(mine, ref, atol=1e-6))         # -> True
print("max abs diff:", (mine - ref).abs().max().item())          # ~1.2e-7

### Step 2 — Roll the cell over a sequenceA single GRU step turns one input plus the previous state into a new state. To process a sequence we just call it repeatedly, feeding each step's output back in as the next step's hidden state, starting from zeros. The final hidden state summarizes the whole sequence — here a length-5 sequence of 3-dim inputs yields a `(2, 4)` state for our batch of 2.

In [ ]:
# Use it in a 2-line net (roll a GRU cell over a sequence).
def gru_seq(X):                                    # X: (batch, T, in_dim)
    h = torch.zeros(X.size(0), hid)
    for t in range(X.size(1)):
        h = my_gru(X[:, t, :], h)
    return h

final_state = gru_seq(torch.randn(2, 5, in_dim))
print("final state shape:", tuple(final_state.shape))

### Step 3 — Recompute the scalar worked exampleThe lesson works one GRU step by hand with scalars and no biases. We reproduce it here so the abstract equations turn into concrete numbers: the reset and update gates are sigmoids of linear mixes, the candidate is a tanh that sees the old state only through `r`, and the blend leaves the state near 0.5 because the update gate `z` is high. These should match the lesson's `r=0.5866`, `z=0.7109`, `h_tilde=0.8066`, `h_new=0.5886`.

In [ ]:
# Recompute the WORKED EXAMPLE (scalar, no bias) and check the lesson numbers.
import math

sig = lambda v: 1 / (1 + math.exp(-v))
x_s = 1.0       # the scalar input
h_p = 0.5       # the previous state

r = sig(0.5 * x_s + (-0.3) * h_p)                 # reset gate   -> 0.5866
z = sig(0.8 * x_s + 0.2 * h_p)                    # update gate  -> 0.7109
ht = math.tanh(1.0 * x_s + 0.4 * (r * h_p))       # candidate    -> 0.8066
hn = (1 - z) * ht + z * h_p                        # blended new state -> 0.5886
print(f"worked: r={r:.4f} z={z:.4f} h_tilde={ht:.4f} h_new={hn:.4f}")

## Visualize the data & results_Give a tanh RNN and a GRU the same task — a +1/-1 cue at step 0, then a run of blanks, recall the cue at the end. As we stretch the gap, which fails first?_

### Step 1 — Build the memory task and a swappable RNN/GRU modelThe task is a pure memory test: a +1/-1 cue appears at step 0, then `T-1` blank steps follow, and the model must recall the cue at the very end. We build a batch generator for it, plus a small model whose recurrent core is either a plain tanh `nn.RNN` or an `nn.GRU` — same everything else — so we can compare the two cell types fairly.

In [ ]:
import torch
import torch.nn as nn

def make_batch(n, T, seed):
    g = torch.Generator().manual_seed(seed)
    bit = torch.randint(0, 2, (n,), generator=g)
    seq = torch.zeros(n, T, 1)
    seq[:, 0, 0] = bit.float() * 2 - 1            # cue at step 0, blanks after
    return seq, bit

class Net(nn.Module):
    def __init__(self, kind, hid=16):
        super().__init__()
        if kind == 'rnn':
            self.rnn = nn.RNN(1, hid, batch_first=True, nonlinearity='tanh')
        else:
            self.rnn = nn.GRU(1, hid, batch_first=True)
        self.fc = nn.Linear(hid, 2)

    def forward(self, x):
        out, _ = self.rnn(x)
        return self.fc(out[:, -1])                # classify from the final state

### Step 2 — Train and sweep the gap lengthWe train each model on the task, then test accuracy as we stretch the gap `T` from 5 up to 35. At short gaps both succeed — a plain RNN can carry one bit a few steps. But as the gap grows, the tanh RNN's signal fades and it collapses to a coin flip (~0.5), while the GRU's update gate can hold `z≈1` to copy the cue forward almost unchanged, staying at 1.0. The split, not the exact numbers, is the reproducible effect.

In [ ]:
def train_eval(kind, T, epochs=400):
    torch.manual_seed(0)
    net = Net(kind)
    opt = torch.optim.Adam(net.parameters(), lr=0.01)
    lossf = nn.CrossEntropyLoss()
    Xtr, ytr = make_batch(256, T, 1)
    for _ in range(epochs):
        opt.zero_grad()
        loss = lossf(net(Xtr), ytr)
        loss.backward()
        opt.step()
    Xte, yte = make_batch(512, T, 2)
    with torch.no_grad():
        acc = (net(Xte).argmax(1) == yte).float().mean().item()
    return round(acc, 3)

for T in [5, 15, 25, 35]:
    rnn_acc = train_eval('rnn', T)
    gru_acc = train_eval('gru', T)
    print(f"T={T:2d}: RNN={rnn_acc:.3f}  GRU={gru_acc:.3f}")
# T= 5: RNN=1.000  GRU=1.000
# T=15: RNN=1.000  GRU=1.000
# T=25: RNN=1.000  GRU=1.000
# T=35: RNN=0.514  GRU=1.000

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** Compute one scalar GRU step (PyTorch blend) with $x=1.0$, $h^{<t-1>}=0.5$, $W_r=0.5,U_r=-0.3$, $W_z=0.8,U_z=0.2$, $W=1.0,U=0.4$, no biases. Give $r,z,\tilde{h},h^{<t>}$.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- $r=\sigma(0.5\cdot1-0.3\cdot0.5)=\sigma(0.35)$. — _Reset gate: linear mix of input and old state, squashed into $[0,1]$._
- $z=\sigma(0.8\cdot1+0.2\cdot0.5)=\sigma(0.90)$. — _Update gate, separate weights._
- $\tilde{h}=\tanh(1.0\cdot1+0.4\cdot(r\cdot0.5))$. — _Candidate sees the old state only through $r\odot h$._
- $h^{<t>}=(1-z)\tilde{h}+z\cdot0.5$. — _PyTorch blend; $z$ near 1 keeps the old state._

**Answer:** $r=0.5866$, $z=0.7109$, $\tilde{h}=0.8066$, $h^{<t>}=0.5886$. Since $z\approx0.71$ is high, the state barely moves from $0.5$.

</details>

**Problem 2.** ABLATION (long-gap memory). We train a tanh RNN and a GRU on the same task: a +1/-1 cue at step 0, then $T-1$ blank steps, recall the cue at the end. We sweep the gap $T$. In our run (gaps 5, 15, 25, 35) the RNN scored [1.0, 1.0, 1.0, 0.514] and the GRU scored [1.0, 1.0, 1.0, 1.0]. What does this show, and which mechanism explains the GRU's last value?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Compare the two models at short gaps. — _At $T\le25$ both are perfect — the gap is short enough that even a plain RNN carries one bit._
- Look at the longest gap $T=35$. — _The RNN collapses to $0.514$ (coin-flip) while the GRU stays at $1.0$ — the qualitative split the paper's long-sentence analysis predicts._
- Attribute the GRU's success. — _When the update gate $z\approx1$, $h^{<t>}\approx h^{<t-1>}$: the cue is copied forward almost unchanged across all 35 blanks, and the gradient rides that near-identity path back undamped._

**Answer:** Both cells handle short gaps; only the GRU survives the long gap (RNN $\to$ chance at $T=35$, GRU $=1.0$). The update gate's near-identity carry ($z\approx1$) is what preserves the bit and the gradient across the gap. (Numbers are our small toy run, seed 0 — not the paper's reported figures; the reproducible effect is the split, not the exact step.)

</details>

**Problem 3.** Why does breaking the gate ordering (slicing $[z,r,n]$ instead of $[r,z,n]$) usually still produce sensible-looking outputs but fail torch.allclose?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Note $r$ and $z$ have identical functional form (sigmoid of a linear map). — _Swapping them still yields valid gates in $[0,1]$, so nothing crashes and outputs look reasonable._
- But the learned weight blocks differ. — _PyTorch's first block was trained/initialised as the reset weights; using it as the update gate changes which numbers multiply where._

**Answer:** The shapes and ranges all match, so you get plausible numbers — but you've fed the reset weights into the update slot and vice-versa, so the result differs from nn.GRUCell and allclose returns False. The lesson: matching a primitive means matching its exact weight packing, not just its formula shape.

</details>